In [1]:
import pandas as pd
import CUSUM_FILTER as cf
import matplotlib.pyplot as plt
import numpy as np
import TB_LABELING as tb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
import KFOLD as kf
import BET_SIZE as bs

In [2]:
df = pd.read_csv("/workspaces/XFUND/AEX.csv", parse_dates=['Date'], index_col='Date')


In [3]:
#BB

period = 20

#Calculate 20-ma and std dev
df['ma_20'] = df['Close'].rolling(period).mean()
df['std'] = df['Close'].rolling(period).std()

#Calculate bollinger bands using std dev
df['upper_bollinger'] = df['ma_20'] + (2 * df['std'])
df['lower_bollinger'] = df['ma_20'] - (2 * df['std'])



#RSI

def gain(value):
    if value < 0:
        return 0
    else:
        return value
    
def loss(value):
    if value > 0:
        return 0
    else:
        return abs(value)
period = 6

#Calculate price delta
df['delta'] = df['Close'].diff()

#Classify delta into gain & loss
df['gain'] = df['delta'].apply(lambda x:gain(x))
df['loss'] = df['delta'].apply(lambda x:loss(x))

#Calculate ema 
df['ema_gain'] = df['gain'].ewm(period).mean()
df['ema_loss'] = df['loss'].ewm(period).mean()

#Calculate RSI
df['rs'] = df['ema_gain']/df['ema_loss']
df['rsi'] = df['rs'].apply(lambda x: 100 - (100/(x+1)))

df['ma_20'] = df['Close'].rolling(window=20).mean()
df.dropna(inplace=True)


#buy signal
df['y_pred_pri'] = np.where(
    (df['rsi'] < 30) &
    (df['Close'] < df['lower_bollinger']), 1, np.nan)

#sell signal
df['y_pred_pri'] = np.where(
    (df['rsi'] > 70) & 
    (df['Close'] > df['upper_bollinger']), -1, df['y_pred_pri'])

#buy/sell next trading day
df['y_pred_pri'] = df['y_pred_pri'].shift()
df['y_pred_pri'] = df['y_pred_pri'].fillna(0)

buy_signal_count = (df['y_pred_pri'] == 1).sum()
sell_signal_count = (df['y_pred_pri'] == -1).sum()
no_signal_count = (df['y_pred_pri'] == 0).sum()

print(f"Buy signals: {buy_signal_count}")
print(f"Sell signals: {sell_signal_count}")
print(f"No signals: {no_signal_count}")


Buy signals: 337
Sell signals: 263
No signals: 6399


In [4]:
close = df["Close"]
display(df)

vol = cf.getDailyVol(close,60)
cusum_events = cf.getTEvents(close, vol.mean()*.1)
cusum_events_aligned = cusum_events[cusum_events >= vol.index[0]]
trgt = vol.loc[cusum_events_aligned]
ptsl = [4,1]

vertical_barriers = tb.add_vertical_barrier(cusum_events_aligned, close, num_days=1)
meta_events = tb.get_events(close, cusum_events_aligned, ptsl , trgt, 0.001, 5, vertical_barriers, side=df['y_pred_pri'])
meta_labels =  tb.get_bins(meta_events, close)
print(meta_labels['bin'].value_counts())

t1 = meta_events['t1'].copy()


# Select the features from df and drop rows with NaN values
X = df[['Close', 'ma_20', 'std', 'upper_bollinger', 'lower_bollinger', 'rsi']]
# Align X with meta_labels by selecting rows from X that exist in meta_labels
X_aligned = X.loc[meta_labels.index]

# Now we can scale X_aligned
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_aligned)

# Convert the scaled data back into a DataFrame
X_final = pd.DataFrame(X_scaled, index=X_aligned.index, columns=X_aligned.columns)

# Split the indices for training and testing
indices_train, indices_test = train_test_split(meta_labels.index, test_size=0.20, shuffle=False)

# Now, use these indices to select the rows from X_final
X_train_full = X_final.loc[indices_train]
X_test_full = X_final.loc[indices_test]

# Select the corresponding meta_labels for training and testing
y_train_full = meta_labels['bin'].loc[indices_train]
y_test_full = meta_labels['bin'].loc[indices_test]

# Now you can use these indices to create train_i and test_i for the getTrainTimes function
train_i = t1.loc[indices_train]
test_i = t1.loc[indices_test]
train_times = kf.getTrainTimes(train_i, test_i)

# Filter the features and labels for training to only include non-overlapping times
X_train_filtered = X_train_full.loc[train_times.index]
y_train_filtered = y_train_full.loc[train_times.index]

# Initialize the model
svc = SVC(probability=True, kernel='linear', random_state=42)

# Initialize the Bagging ensemble classifier, using SVC as the base estimator
svc_ensemble = BaggingClassifier(estimator=svc, n_estimators=100, random_state=42)

# Fit the model on the filtered dataset
svc_ensemble.fit(X_train_filtered, y_train_filtered)

# Make predictions on the test set
y_pred_proba = svc_ensemble.predict_proba(X_test_full)

# Extract the probabilities for the positive class
probabilities = y_pred_proba[:, 1]  # probabilities now holds an array of probabilities of the positive class

# Calculate the moving average and current price

current_price = df['Close'].iloc[-1]
divergence = (current_price - df['ma_20'].iloc[-1]) / df['ma_20'].iloc[-1]
m = 0.5  # Adjust based on strategy optimization

# Calculate the weight using the getW function
w = bs.getW(divergence, m)

# Maximum position size defined by your strategy
maxPos = 50  # Example maximum position size
# Using the current moving average as the mean price
mP = df['ma_20'].iloc[-1]

# Calculate the target position size
tPos = bs.getTPos(w, current_price, mP, maxPos)

# Calculate the bet size for each probability prediction
bet_sizes = [bs.betSize(w, p - mP) for p in probabilities]





,Open,High,Low,Close,Adj Close,Volume,ma_20,std,upper_bollinger,lower_bollinger,delta,gain,loss,ema_gain,ema_loss,rs,rsi,y_pred_pri
Date,,,,,,,,,,,,,,,,,,
1992-11-06,125.434494,126.242226,125.175842,126.015335,126.015335,0.0,125.822707,0.833921,127.490549,124.154865,0.626221,0.626221,0.000000,0.385320,0.328562,1.172746,53.975297,0.0
1992-11-09,125.879204,126.750465,125.870125,126.242226,126.242226,0.0,125.787538,0.798137,127.383813,124.191263,0.226891,0.226891,0.000000,0.361600,0.279370,1.294339,56.414468,0.0
1992-11-10,126.364754,126.990967,126.201385,126.759544,126.759544,0.0,125.783001,0.792012,127.367025,124.198977,0.517318,0.517318,0.000000,0.384755,0.237829,1.617781,61.799708,0.0
1992-11-11,126.237694,126.664246,126.042564,126.205925,126.205925,0.0,125.768933,0.781270,127.331474,124.206393,-0.553619,0.000000,0.553619,0.327875,0.284513,1.152407,53.540386,0.0
1992-11-12,126.818535,127.980217,126.555336,127.594498,127.594498,0.0,125.857421,0.881693,127.620806,124.094036,1.388573,1.388573,0.000000,0.483906,0.242661,1.994164,66.601700,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-10-13,738.820007,743.000000,733.080017,733.900024,733.900024,54300.0,729.337500,6.878650,743.094801,715.580200,-7.619996,0.000000,7.619996,2.366827,2.000183,1.183305,54.197888,0.0
2023-10-16,736.809998,737.559998,731.500000,735.390015,735.390015,44000.0,729.314502,6.856464,743.027429,715.601575,1.489991,1.489991,0.000000,2.241565,1.714442,1.307460,56.662304,0.0
2023-10-17,733.450012,738.429993,730.500000,736.770020,736.770020,43800.0,729.406503,6.947519,743.301542,715.511464,1.380005,1.380005,0.000000,2.118485,1.469522,1.441615,59.043501,0.0


2023-11-09 12:00:01.496691 100.0% apply_pt_sl_on_t1 done after 0.1 minutes. Remaining 0.0 minutes.


bin
0    6920
1      67
Name: count, dtype: int64


In [9]:
import empyrical

# Assuming cf, tb, and kf are defined as before and contain necessary functions for cusum filter, triple barrier, and cross-validation, respectively
def returns_series(xtest, sd, times, close, filtered):
    
    ret_df = pd.DataFrame()
    
    # Assuming sd[0] is a DataFrame with the same index as xtest that contains the side (-1/1)
    ret_df['ret_side'] = sd[0].loc[xtest.index]
    
    ret_df['bet_pass'] = filtered.copy()
    
    # Keep only the trades that pass the filter
    ret_df = ret_df[ret_df.bet_pass != 0]
    
    # Create separate DataFrames for buy and sell operations based on the side
    p_buy = pd.DataFrame(index=ret_df[ret_df.ret_side == 1].index)
    p_sell = pd.DataFrame(index=ret_df[ret_df.ret_side == -1].index)
    
    # For both buy and sell, the price at which the operation occurs is the close price
    p_buy['price'] = close.copy()
    p_sell['price'] = close.copy()
    
    # Store the times for the exit of the positions
    p_buy['stop'] = times.copy()
    p_sell['stop'] = times.copy()
    
    # Get the closing price at the stop times for both buy and sell
    val_close_at_stop = close.loc[p_buy['stop'].append(p_sell['stop'])]
    
    # We no longer need the 'stop' column
    p_buy.drop(columns=['stop'], inplace=True)
    p_sell.drop(columns=['stop'], inplace=True)
    
    # Assign the closing price at the stop times to the end_price for both buy and sell
    p_buy['end_price'] = val_close_at_stop.loc[p_buy.index].values
    p_sell['end_price'] = val_close_at_stop.loc[p_sell.index].values
    
    # Calculate the returns for buys and sells
    g = (p_buy['end_price'] - p_buy['price']) / p_buy['price']
    g2 = (p_sell['end_price'] - p_sell['price']) / p_sell['price'] * -1
    
    # Concatenate the returns from both buy and sell into a single Series
    returns_series = pd.concat([g, g2])
    
    return returns_series

def backtest_strategy(df, n_splits=10, embargo=0.01):
    #BB

    period = 20

    #Calculate 20-ma and std dev
    df['ma_20'] = df['Close'].rolling(period).mean()
    df['std'] = df['Close'].rolling(period).std()

    #Calculate bollinger bands using std dev
    df['upper_bollinger'] = df['ma_20'] + (2 * df['std'])
    df['lower_bollinger'] = df['ma_20'] - (2 * df['std'])



    #RSI

    def gain(value):
        if value < 0:
            return 0
        else:
            return value
        
    def loss(value):
        if value > 0:
            return 0
        else:
            return abs(value)
    period = 6

    #Calculate price delta
    df['delta'] = df['Close'].diff()

    #Classify delta into gain & loss
    df['gain'] = df['delta'].apply(lambda x:gain(x))
    df['loss'] = df['delta'].apply(lambda x:loss(x))

    #Calculate ema 
    df['ema_gain'] = df['gain'].ewm(period).mean()
    df['ema_loss'] = df['loss'].ewm(period).mean()

    #Calculate RSI
    df['rs'] = df['ema_gain']/df['ema_loss']
    df['rsi'] = df['rs'].apply(lambda x: 100 - (100/(x+1)))

    df['ma_20'] = df['Close'].rolling(window=20).mean()
    df.dropna(inplace=True)


    #buy signal
    df['y_pred_pri'] = np.where(
        (df['rsi'] < 30) &
        (df['Close'] < df['lower_bollinger']), 1, np.nan)

    #sell signal
    df['y_pred_pri'] = np.where(
        (df['rsi'] > 70) & 
        (df['Close'] > df['upper_bollinger']), -1, df['y_pred_pri'])

    #buy/sell next trading day
    df['y_pred_pri'] = df['y_pred_pri'].shift()
    df['y_pred_pri'] = df['y_pred_pri'].fillna(0)
    
    vol = cf.getDailyVol(close,60)
    cusum_events = cf.getTEvents(close, vol.mean()*.1)
    cusum_events_aligned = cusum_events[cusum_events >= vol.index[0]]
    trgt = vol.loc[cusum_events_aligned]
    ptsl = [4,1]

    vertical_barriers = tb.add_vertical_barrier(cusum_events_aligned, close, num_days=1)
    meta_events = tb.get_events(close, cusum_events_aligned, ptsl , trgt, 0.001, 5, vertical_barriers, side=df['y_pred_pri'])
    meta_labels =  tb.get_bins(meta_events, close)
    print(meta_labels['bin'].value_counts())
    t1 = meta_events['t1'].copy()
    # Prepare for backtesting
    
    backtest_df = pd.DataFrame()
    t1_aligned = t1.reindex(X_final.index)

    cvGen = kf.PurgedKFold(n_splits=n_splits, t1=t1_aligned, pctEmbargo=embargo)
    i = 1
    for train, test in cvGen.split(X=X_final):
        X_train, y_train = X_final.iloc[train], y_train_full.iloc[train]
        X_test, y_test = X_final.iloc[test], y_test_full.iloc[test]

        # Initialize and fit the model
        svc = SVC(probability=True, kernel='linear', random_state=42)
        svc_ensemble = BaggingClassifier(estimator=svc, n_estimators=100, random_state=42)
        svc_ensemble.fit(X_train, y_train)

        # Predict on the test set
        y_pred_proba = svc_ensemble.predict_proba(X_test)
        
        probabilities = y_pred_proba[:, 1]  # probabilities of the positive class
        current_price = df['Close'].iloc[-1]
        divergence = (current_price - df['ma_20'].iloc[-1]) / df['ma_20'].iloc[-1]
        m = 0.5  # Adjust based on strategy optimization

        # Calculate the weight using the getW function
        w = bs.getW(divergence, m)

        # Maximum position size defined by your strategy
        maxPos = 50  # Example maximum position size
        # Using the current moving average as the mean price
        mP = df['ma_20'].iloc[-1]

        # Calculate the target position size
        tPos = bs.getTPos(w, current_price, mP, maxPos)

        # Calculate the bet size for each probability prediction
        bet_sizes = [bs.betSize(w, p - mP) for p in probabilities]

        # Assuming you have a function to calculate returns similar to returns_series provided earlier
        returns = returns_series(X_test, probabilities, t1_aligned[test], df['Close'].iloc[test], bet_sizes)

        # Calculate performance metrics
        backtest_df.at[i, 'cum_returns'] = np.around(empyrical.stats.cum_returns_final(returns) * 100, 2)
        backtest_df.at[i, 'sharpe_ratio'] = np.around(empyrical.stats.sharpe_ratio(returns), 2)
        backtest_df.at[i, 'max_drawdown'] = np.around(empyrical.stats.max_drawdown(returns) * 100, 2)
        i += 1

    return backtest_df

# Now you can call this function with your DataFrame to start the backtesting
results_df = backtest_strategy(df)
print(results_df)


KeyError: "[Timestamp('1992-11-10 00:00:00'), Timestamp('1992-11-11 00:00:00'), Timestamp('1992-11-12 00:00:00'), Timestamp('1992-11-13 00:00:00'), Timestamp('1992-11-16 00:00:00'), Timestamp('1992-11-17 00:00:00'), Timestamp('1992-11-18 00:00:00'), Timestamp('1992-11-19 00:00:00'), Timestamp('1992-11-20 00:00:00'), Timestamp('1992-11-23 00:00:00'), Timestamp('1992-11-24 00:00:00'), Timestamp('1992-11-25 00:00:00'), Timestamp('1992-11-26 00:00:00'), Timestamp('1992-11-27 00:00:00'), Timestamp('1992-11-30 00:00:00'), Timestamp('1992-12-01 00:00:00'), Timestamp('1992-12-02 00:00:00'), Timestamp('1992-12-03 00:00:00'), Timestamp('1992-12-04 00:00:00'), Timestamp('1992-12-07 00:00:00'), Timestamp('1992-12-08 00:00:00'), Timestamp('1992-12-09 00:00:00'), Timestamp('1992-12-10 00:00:00'), Timestamp('1992-12-11 00:00:00'), Timestamp('1992-12-14 00:00:00'), Timestamp('1992-12-15 00:00:00'), Timestamp('1992-12-16 00:00:00'), Timestamp('1992-12-17 00:00:00'), Timestamp('1992-12-18 00:00:00'), Ti